# Week 9: Feature Engineering & Preprocessing Pipeline
# 第 9 週：特徵工程與資料前處理管線

## 學習目標 Learning Objectives
- 比較不同類別特徵編碼方法 (OneHot, Ordinal, Label)
- 視覺化三種 Scaler (Standard, MinMax, Robust) 的效果差異
- 理解缺失值處理策略 (Simple, KNN Imputation) 及其影響
- 掌握特徵轉換 (Log, Power, Box-Cox) 的前後分布變化
- 使用 `sklearn.pipeline.Pipeline` 與 `make_pipeline` 建構前處理流程
- 用 `ColumnTransformer` 對不同類型特徵分別處理
- 整合前處理與模型訓練為完整 Pipeline 並交叉驗證
- 實作自定義 Transformer

In [ ]:
# 環境準備 Environment Setup
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing, load_iris
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, OrdinalEncoder, LabelEncoder,
    PowerTransformer, PolynomialFeatures
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import accuracy_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 設定中文字型與繪圖風格
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

np.random.seed(42)
print('所有套件載入成功！All packages imported successfully!')

In [ ]:
# 建立混合型資料集 Create a mixed dataset (numeric + categorical + missing values)
np.random.seed(42)
n = 500

# 數值特徵 Numeric features
age = np.random.normal(40, 12, n).clip(18, 80)
income = np.random.exponential(50000, n)  # 右偏分布
credit_score = np.random.normal(650, 80, n).clip(300, 850)
hours_per_week = np.random.normal(40, 10, n).clip(10, 80)

# 類別特徵 Categorical features
education_levels = ['High School', 'Bachelor', 'Master', 'PhD']
education = np.random.choice(education_levels, n, p=[0.3, 0.4, 0.2, 0.1])

city_list = ['Taipei', 'Taichung', 'Kaohsiung', 'Tainan', 'Hsinchu']
city = np.random.choice(city_list, n, p=[0.35, 0.2, 0.2, 0.15, 0.1])

gender_list = ['M', 'F']
gender = np.random.choice(gender_list, n)

# 建立目標變數 (二元分類: 是否核准貸款)
edu_map = {'High School': 0, 'Bachelor': 1, 'Master': 2, 'PhD': 3}
edu_numeric = np.array([edu_map[e] for e in education])
prob = 1 / (1 + np.exp(-(0.02 * (credit_score - 650) + 0.3 * edu_numeric
                          + 0.00001 * income - 0.01 * age + 0.5)))
target = (np.random.rand(n) < prob).astype(int)

# 注入缺失值 Inject missing values (~10%)
age_missing = age.copy()
income_missing = income.copy()
credit_missing = credit_score.copy()

missing_mask_age = np.random.rand(n) < 0.08
missing_mask_income = np.random.rand(n) < 0.12
missing_mask_credit = np.random.rand(n) < 0.06
age_missing[missing_mask_age] = np.nan
income_missing[missing_mask_income] = np.nan
credit_missing[missing_mask_credit] = np.nan

print(f'資料集大小: {n} 筆')
print(f'數值特徵: age, income, credit_score, hours_per_week')
print(f'類別特徵: education, city, gender')
print(f'目標變數: loan_approved (0/1)')
print(f'\n缺失值統計:')
print(f'  age:          {missing_mask_age.sum()} ({missing_mask_age.mean()*100:.1f}%)')
print(f'  income:       {missing_mask_income.sum()} ({missing_mask_income.mean()*100:.1f}%)')
print(f'  credit_score: {missing_mask_credit.sum()} ({missing_mask_credit.mean()*100:.1f}%)')
print(f'\n目標分布: Approved={target.sum()}, Rejected={n-target.sum()}')

---
## Part 1: Encoding Categorical Variables — 類別特徵編碼

機器學習模型通常需要數值輸入，因此類別特徵需要編碼轉換。

| 方法 | 適用場景 | 優點 | 缺點 |
|------|---------|------|------|
| **LabelEncoder** | 有序類別 / 樹模型 | 不增加維度 | 引入虛假的大小順序 |
| **OrdinalEncoder** | 有序類別 | 可指定順序 | 同上 |
| **OneHotEncoder** | 名義類別 | 不引入順序 | 高基數時維度爆炸 |

In [ ]:
# Part 1: Encoding Comparison
print('=== LabelEncoder ===')
le = LabelEncoder()
edu_label = le.fit_transform(education)
print(f'Classes: {le.classes_}')
print(f'Encoded (first 10): {edu_label[:10]}')
print(f'Issue: PhD=3 > Master=2 > High School=1 > Bachelor=0 (alphabetical, not logical!)\n')

print('=== OrdinalEncoder ===')
oe = OrdinalEncoder(categories=[['High School', 'Bachelor', 'Master', 'PhD']])
edu_ordinal = oe.fit_transform(education.reshape(-1, 1))
print(f'Encoded (first 10): {edu_ordinal[:10].ravel().astype(int)}')
print(f'Correct order: High School=0 < Bachelor=1 < Master=2 < PhD=3\n')

print('=== OneHotEncoder ===')
ohe = OneHotEncoder(sparse_output=False)
city_onehot = ohe.fit_transform(city.reshape(-1, 1))
print(f'Categories: {ohe.categories_[0]}')
print(f'Shape: {city.reshape(-1, 1).shape} -> {city_onehot.shape}')
print(f'First row (city={city[0]}): {city_onehot[0]}')

In [ ]:
# 視覺化編碼結果比較
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# LabelEncoder
le_vals = LabelEncoder().fit_transform(education)
unique_labels, counts_l = np.unique(le_vals, return_counts=True)
le_temp = LabelEncoder()
le_temp.fit(education)
axes[0].bar(unique_labels, counts_l, color='#3b82f6', edgecolor='black', alpha=0.8)
axes[0].set_xticks(unique_labels)
axes[0].set_xticklabels([f'{le_temp.classes_[i]}\n({i})' for i in unique_labels], fontsize=9)
axes[0].set_title('LabelEncoder\n(Alphabetical Order)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].grid(True, alpha=0.3, axis='y')

# OrdinalEncoder
axes[1].bar(range(4), [np.sum(edu_ordinal == i) for i in range(4)],
            color='#10b981', edgecolor='black', alpha=0.8)
axes[1].set_xticks(range(4))
axes[1].set_xticklabels(['High School\n(0)', 'Bachelor\n(1)',
                          'Master\n(2)', 'PhD\n(3)'], fontsize=9)
axes[1].set_title('OrdinalEncoder\n(Custom Logical Order)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].grid(True, alpha=0.3, axis='y')

# OneHotEncoder
ohe_categories = ohe.categories_[0]
city_counts = [np.sum(city == c) for c in ohe_categories]
x_pos = np.arange(len(ohe_categories))
axes[2].bar(x_pos, city_counts, color='#f59e0b', edgecolor='black', alpha=0.8)
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(ohe_categories, fontsize=9, rotation=15)
axes[2].set_title(f'OneHotEncoder\n(5 categories -> {city_onehot.shape[1]} columns)',
                  fontsize=13, fontweight='bold')
axes[2].set_ylabel('Count')
axes[2].grid(True, alpha=0.3, axis='y')

# 在上方標示 OneHot 向量範例
for i, cat in enumerate(ohe_categories):
    vec = [0]*len(ohe_categories)
    vec[i] = 1
    axes[2].text(i, city_counts[i] + 3, str(vec), ha='center', fontsize=6,
                color='#7c3aed', fontweight='bold')

fig.suptitle('類別特徵編碼方法比較\nCategorical Encoding Methods Comparison',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

---
## Part 2: Feature Scaling Comparison — 特徵縮放

**StandardScaler (標準化):**
$$z = \frac{x - \mu}{\sigma}$$

**MinMaxScaler (最小-最大正規化):**
$$x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

**RobustScaler (穩健縮放):**
$$x_{\text{robust}} = \frac{x - Q_{50}}{Q_{75} - Q_{25}}$$

In [ ]:
# Part 2: Feature Scaling Comparison
# 使用 income（右偏分布 + 離群值）來展示差異
income_clean = income.copy()
# 加入幾個離群值
income_with_outliers = income_clean.copy()
income_with_outliers[0] = 500000  # 離群值
income_with_outliers[1] = 450000
income_with_outliers[2] = 600000

data_to_scale = income_with_outliers.reshape(-1, 1)

scalers = [
    ('Original', None),
    ('StandardScaler', StandardScaler()),
    ('MinMaxScaler', MinMaxScaler()),
    ('RobustScaler', RobustScaler()),
]

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
colors_sc = ['#6366f1', '#3b82f6', '#10b981', '#f59e0b']

for ax, (name, scaler), color in zip(axes, scalers, colors_sc):
    if scaler is None:
        scaled = data_to_scale.ravel()
    else:
        scaled = scaler.fit_transform(data_to_scale).ravel()

    ax.hist(scaled, bins=50, color=color, edgecolor='black', alpha=0.7)
    ax.axvline(x=np.mean(scaled), color='red', linestyle='--', linewidth=2,
               label=f'Mean={np.mean(scaled):.2f}')
    ax.axvline(x=np.median(scaled), color='green', linestyle=':', linewidth=2,
               label=f'Median={np.median(scaled):.2f}')
    ax.set_title(f'{name}\nRange=[{scaled.min():.2f}, {scaled.max():.2f}]',
                fontsize=12, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('特徵縮放方法比較 (含離群值的收入資料)\n'
             'Scaler Comparison on Income Data with Outliers',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print('觀察：')
print('- StandardScaler: 離群值被保留，均值=0，但範圍不固定')
print('- MinMaxScaler: 離群值壓縮了大部分資料到很小的範圍 -> 最敏感')
print('- RobustScaler: 使用中位數和 IQR，對離群值最穩健')

In [ ]:
# 多特徵同時縮放比較
features_to_scale = np.column_stack([age, income_with_outliers, credit_score, hours_per_week])
feat_labels = ['Age', 'Income', 'Credit Score', 'Hours/Week']

fig, axes = plt.subplots(3, 1, figsize=(14, 12))
scaler_list = [
    ('StandardScaler', StandardScaler()),
    ('MinMaxScaler', MinMaxScaler()),
    ('RobustScaler', RobustScaler()),
]

for ax, (sname, scaler) in zip(axes, scaler_list):
    scaled_data = scaler.fit_transform(features_to_scale)
    bp = ax.boxplot([scaled_data[:, i] for i in range(4)],
                     labels=feat_labels, patch_artist=True,
                     boxprops=dict(linewidth=1.5),
                     medianprops=dict(color='black', linewidth=2))

    box_colors = ['#3b82f6', '#10b981', '#f59e0b', '#8b5cf6']
    for patch, color in zip(bp['boxes'], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)

    ax.set_title(f'{sname}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Scaled Value')
    ax.grid(True, alpha=0.3, axis='y')
    ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)

fig.suptitle('三種 Scaler 對多特徵的縮放效果比較\nScaler Effects on Multiple Features',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('注意 Income 特徵在 MinMaxScaler 下的箱型圖被壓縮 -> 離群值效應')

---
## Part 3: Handling Missing Values — 缺失值處理

缺失值處理策略：
- **SimpleImputer(strategy='mean')**: 用平均值填補
- **SimpleImputer(strategy='median')**: 用中位數填補（對離群值穩健）
- **SimpleImputer(strategy='most_frequent')**: 用眾數填補
- **KNNImputer**: 用 K 最近鄰居的加權平均填補（考慮特徵間相關性）

In [ ]:
# Part 3: Missing Value Imputation Comparison
X_with_missing = np.column_stack([age_missing, income_missing, credit_missing, hours_per_week])
X_complete = np.column_stack([age, income, credit_score, hours_per_week])

print(f'缺失值總數: {np.isnan(X_with_missing).sum()}')
print(f'各欄位缺失: {np.isnan(X_with_missing).sum(axis=0)}')

# 比較不同填補策略
imputers = [
    ('Mean', SimpleImputer(strategy='mean')),
    ('Median', SimpleImputer(strategy='median')),
    ('KNN (k=5)', KNNImputer(n_neighbors=5)),
]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 只看 income 欄位（有較多缺失值且右偏）
income_col_idx = 1
income_missing_mask = np.isnan(X_with_missing[:, income_col_idx])

for ax, (name, imputer) in zip(axes, imputers):
    X_imputed = imputer.fit_transform(X_with_missing)

    # 原始有值的分布
    ax.hist(X_with_missing[~income_missing_mask, income_col_idx], bins=40,
            alpha=0.5, color='#3b82f6', edgecolor='black', label='Original (non-missing)',
            density=True)
    # 填補後的值
    ax.hist(X_imputed[income_missing_mask, income_col_idx], bins=15,
            alpha=0.7, color='#dc2626', edgecolor='black', label='Imputed values',
            density=True)

    imputed_vals = X_imputed[income_missing_mask, income_col_idx]
    ax.set_title(f'{name}\nImputed mean={np.mean(imputed_vals):.0f}, '
                 f'std={np.std(imputed_vals):.0f}',
                fontsize=12, fontweight='bold')
    ax.set_xlabel('Income')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle('缺失值填補策略比較 (Income 欄位)\nImputation Strategy Comparison',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print('觀察：')
print('- Mean/Median: 所有缺失值被填入同一個值 -> 在直方圖上形成尖峰')
print('- KNN: 根據鄰居的值填補 -> 填補值的分布較自然')

In [ ]:
# 填補策略對下游模型效能的影響
# 先準備完整的特徵矩陣（數值+編碼後的類別）
edu_encoded = OrdinalEncoder(categories=[['High School', 'Bachelor', 'Master', 'PhD']]).fit_transform(
    education.reshape(-1, 1))
gender_encoded = LabelEncoder().fit_transform(gender).reshape(-1, 1)

imputer_results = {}
for name, imputer in imputers:
    X_imp = imputer.fit_transform(X_with_missing)
    X_combined = np.hstack([X_imp, edu_encoded, gender_encoded])

    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    scores = cross_val_score(clf, X_combined, target, cv=5, scoring='accuracy')
    imputer_results[name] = scores
    print(f'{name:15s}: CV Accuracy = {scores.mean():.4f} (+/- {scores.std():.4f})')

# 基準：使用完整資料（無缺失值）
X_baseline = np.hstack([X_complete, edu_encoded, gender_encoded])
scores_base = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42),
                               X_baseline, target, cv=5, scoring='accuracy')
imputer_results['Complete Data'] = scores_base
print(f'{"Complete Data":15s}: CV Accuracy = {scores_base.mean():.4f} (+/- {scores_base.std():.4f})')

# 箱型圖
fig, ax = plt.subplots(figsize=(9, 5))
bp = ax.boxplot(list(imputer_results.values()), labels=list(imputer_results.keys()),
                patch_artist=True, medianprops=dict(color='black', linewidth=2))
box_cols = ['#3b82f6', '#10b981', '#f59e0b', '#8b5cf6']
for patch, color in zip(bp['boxes'], box_cols):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_ylabel('CV Accuracy')
ax.set_title('填補策略對模型效能的影響\nImputation Impact on Model Performance',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## Part 4: Feature Transformations — 特徵轉換

許多模型假設特徵近似常態分布，但實際資料常有偏態。常見轉換：

- **Log Transform**: $x' = \log(x + 1)$，適用於右偏資料
- **Power Transform (Yeo-Johnson)**: 自動找最佳冪次，可處理負值
- **Box-Cox**: $x' = \frac{x^\lambda - 1}{\lambda}$，僅適用正值

In [ ]:
# Part 4: Feature Transformations — Before/After Distribution
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 使用 income（右偏分布）作為範例
income_sample = income.copy()

transforms = [
    ('Original', income_sample),
    ('Log Transform\n$\\log(x+1)$', np.log1p(income_sample)),
    ('Square Root\n$\\sqrt{x}$', np.sqrt(income_sample)),
]

# 上排：直方圖
colors_t = ['#6366f1', '#3b82f6', '#10b981']
for ax, (name, data), color in zip(axes[0], transforms, colors_t):
    ax.hist(data, bins=50, color=color, edgecolor='black', alpha=0.7)
    skewness = stats.skew(data)
    ax.set_title(f'{name}\nSkewness={skewness:.3f}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.grid(True, alpha=0.3, axis='y')

# 下排：使用 PowerTransformer (Yeo-Johnson)
pt_yj = PowerTransformer(method='yeo-johnson')
income_yj = pt_yj.fit_transform(income_sample.reshape(-1, 1)).ravel()

pt_bc = PowerTransformer(method='box-cox')
income_bc = pt_bc.fit_transform(income_sample.reshape(-1, 1)).ravel()

transforms_2 = [
    ('Yeo-Johnson', income_yj),
    ('Box-Cox', income_bc),
    ('Comparison: Q-Q Plot', None),
]

colors_t2 = ['#f59e0b', '#dc2626', '#6b7280']
for ax, (name, data), color in zip(axes[1, :2], transforms_2[:2], colors_t2[:2]):
    ax.hist(data, bins=50, color=color, edgecolor='black', alpha=0.7)
    skewness = stats.skew(data)
    ax.set_title(f'PowerTransformer ({name})\nSkewness={skewness:.3f}',
                fontsize=12, fontweight='bold')
    ax.set_xlabel('Transformed Value')
    ax.set_ylabel('Count')
    ax.grid(True, alpha=0.3, axis='y')

# Q-Q Plot 比較
ax_qq = axes[1, 2]
for data, name, color in [(income_sample, 'Original', '#6366f1'),
                            (income_yj, 'Yeo-Johnson', '#f59e0b'),
                            (income_bc, 'Box-Cox', '#dc2626')]:
    sorted_data = np.sort(data)
    theoretical = stats.norm.ppf(np.linspace(0.01, 0.99, len(data)))
    # Subsample for clarity
    step = max(1, len(data) // 100)
    ax_qq.scatter(theoretical[::step], sorted_data[::step], s=10, alpha=0.5,
                  label=name, color=color)

ax_qq.plot([-3, 3], [-3, 3], 'k--', linewidth=1, label='45-degree line')
ax_qq.set_xlabel('Theoretical Quantiles')
ax_qq.set_ylabel('Sample Quantiles (standardized)')
ax_qq.set_title('Q-Q Plot Comparison\n(closeness to diagonal = normality)',
               fontsize=12, fontweight='bold')
ax_qq.legend(fontsize=8)
ax_qq.grid(True, alpha=0.3)

fig.suptitle('特徵轉換前後的分布比較\nFeature Transformation: Before vs After',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('觀察：')
print('- 原始 income 高度右偏 (positive skew)')
print('- Log / Sqrt 可以減輕偏態')
print('- Yeo-Johnson / Box-Cox 自動找最佳冪次 -> 接近常態分布')

---
## Part 5: sklearn Pipeline — 建構前處理管線

Pipeline 將多個前處理步驟串聯成一個物件，好處：
1. **避免資料洩漏 (Data Leakage)**: fit 只在訓練集進行
2. **可重現性 (Reproducibility)**: 所有步驟打包為單一物件
3. **簡化部署**: 一個物件即可進行預測

```
X_raw -> Imputer -> Scaler -> Model -> y_pred
```

In [ ]:
# Part 5: 基礎 Pipeline
# 使用只含數值特徵的簡化資料集
X_numeric = X_with_missing.copy()  # age, income, credit_score, hours_per_week (with NaN)

X_n_train, X_n_test, y_n_train, y_n_test = train_test_split(
    X_numeric, target, test_size=0.2, random_state=42)

# 方式 1: 手動逐步處理（容易出錯）
print('=== 方式 1: 手動逐步處理 (Manual Steps) ===')
imp = SimpleImputer(strategy='median')
X_n_train_imp = imp.fit_transform(X_n_train)
X_n_test_imp = imp.transform(X_n_test)  # 用 transform，不是 fit_transform！

sc = StandardScaler()
X_n_train_sc = sc.fit_transform(X_n_train_imp)
X_n_test_sc = sc.transform(X_n_test_imp)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_n_train_sc, y_n_train)
print(f'Test Accuracy: {lr.score(X_n_test_sc, y_n_test):.4f}')

# 方式 2: 使用 Pipeline（更簡潔、更安全）
print('\n=== 方式 2: Pipeline (Recommended) ===')
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

pipe.fit(X_n_train, y_n_train)
print(f'Test Accuracy: {pipe.score(X_n_test, y_n_test):.4f}')

# 方式 3: make_pipeline（自動命名）
print('\n=== 方式 3: make_pipeline ===')
pipe_auto = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression(random_state=42, max_iter=1000)
)
pipe_auto.fit(X_n_train, y_n_train)
print(f'Test Accuracy: {pipe_auto.score(X_n_test, y_n_test):.4f}')

# Pipeline 的步驟名稱
print(f'\nPipeline steps: {[name for name, _ in pipe.steps]}')
print('Pipeline 確保 fit 只在訓練集進行 -> 避免資料洩漏')

---
## Part 6: ColumnTransformer — 分欄位處理

實務中，數值特徵和類別特徵需要不同的處理方式。
`ColumnTransformer` 可以對不同欄位套用不同的轉換管線。

```
                  ┌─ Numeric:   Imputer -> Scaler ─────────────┐
X_raw ──> ColumnTransformer                                     ──> X_transformed
                  └─ Categorical: Imputer -> OneHotEncoder ────┘
```

In [ ]:
# Part 6: ColumnTransformer
# 準備含數值和類別的完整資料矩陣
# 用 object 陣列來混合數值和字串
# 我們需要分開處理

# 數值欄位索引 & 類別欄位
numeric_features_idx = [0, 1, 2, 3]  # age, income, credit, hours
numeric_feature_names = ['age', 'income', 'credit_score', 'hours_per_week']
categorical_feature_names = ['education', 'city', 'gender']

# 數值 Pipeline
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 類別 Pipeline
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 組合成 ColumnTransformer
# 由於我們的資料分散在不同的 numpy 陣列中，先組裝
X_num_full = X_with_missing  # (500, 4) with NaN
X_cat_full = np.column_stack([education, city, gender])  # (500, 3) strings

# 分開傳入 ColumnTransformer 需要統一的矩陣
# 這裡我們示範正確的做法
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, [0, 1, 2, 3]),
        ('cat', categorical_transformer, [4, 5, 6])
    ],
    remainder='drop'
)

# 組合所有特徵為 object 陣列
X_all = np.column_stack([X_with_missing.astype(object), education, city, gender])

X_all_train, X_all_test, y_all_train, y_all_test = train_test_split(
    X_all, target, test_size=0.2, random_state=42)

# 測試 ColumnTransformer
X_transformed_train = preprocessor.fit_transform(X_all_train)
X_transformed_test = preprocessor.transform(X_all_test)

print(f'原始特徵維度: {X_all_train.shape[1]} (4 numeric + 3 categorical)')
print(f'轉換後維度:   {X_transformed_train.shape[1]} '
      f'(4 scaled numeric + {X_transformed_train.shape[1] - 4} one-hot encoded)')
print(f'\nColumnTransformer 結構:')
for name, transformer, columns in preprocessor.transformers_:
    if hasattr(transformer, 'steps'):
        steps_str = ' -> '.join([s[0] for s in transformer.steps])
        print(f'  [{name}] columns={columns}: {steps_str}')
    else:
        print(f'  [{name}] columns={columns}: {transformer}')

---
## Part 7: Full Pipeline with Model — 完整管線

將 ColumnTransformer（前處理）與分類器串聯為完整的 Pipeline，
然後用交叉驗證評估不同模型的效能。

In [ ]:
# Part 7: Full Pipeline = Preprocessor + Classifier
classifiers = [
    ('Logistic Regression', LogisticRegression(random_state=42, max_iter=1000)),
    ('KNN (k=5)', KNeighborsClassifier(n_neighbors=5)),
    ('SVM (RBF)', SVC(kernel='rbf', random_state=42)),
    ('Decision Tree', DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('GBDT', GradientBoostingClassifier(n_estimators=100, random_state=42)),
]

cv_results = {}
print('Full Pipeline: ColumnTransformer + Classifier')
print('=' * 60)

for clf_name, clf in classifiers:
    full_pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])

    scores = cross_val_score(full_pipe, X_all, target, cv=5, scoring='accuracy')
    cv_results[clf_name] = scores
    print(f'{clf_name:25s}: {scores.mean():.4f} (+/- {scores.std():.4f})')

# 箱型圖比較
fig, ax = plt.subplots(figsize=(12, 6))
bp = ax.boxplot(list(cv_results.values()), labels=list(cv_results.keys()),
                patch_artist=True, medianprops=dict(color='black', linewidth=2))

box_colors_7 = ['#3b82f6', '#10b981', '#f59e0b', '#8b5cf6', '#06b6d4', '#dc2626']
for patch, color in zip(bp['boxes'], box_colors_7):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.set_ylabel('CV Accuracy', fontsize=12)
ax.set_title('Full Pipeline + Different Classifiers: 5-Fold CV\n'
             '完整管線 + 不同分類器的交叉驗證比較',
             fontsize=14, fontweight='bold')
ax.set_xticklabels(list(cv_results.keys()), rotation=20, ha='right', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f'\nBest model: {max(cv_results, key=lambda k: cv_results[k].mean())} '
      f'(mean={max(v.mean() for v in cv_results.values()):.4f})')

In [ ]:
# 比較使用 vs 不使用前處理的效能差異
print('Pipeline 的價值：前處理對不同模型的影響\n')

# 不使用前處理（只用數值特徵，填補後直接丟入模型）
X_simple = SimpleImputer(strategy='mean').fit_transform(X_with_missing)

models_compare = [
    ('KNN', KNeighborsClassifier(n_neighbors=5)),
    ('SVM', SVC(kernel='rbf', random_state=42)),
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42)),
]

fig, ax = plt.subplots(figsize=(10, 6))
x_bar = np.arange(len(models_compare))
width = 0.35

no_pipe_scores = []
with_pipe_scores = []

for clf_name, clf in models_compare:
    # Without pipeline (numeric only, no scaling)
    s1 = cross_val_score(clf, X_simple, target, cv=5, scoring='accuracy').mean()
    no_pipe_scores.append(s1)

    # With full pipeline
    s2 = cv_results[clf_name].mean() if clf_name in cv_results else s1
    with_pipe_scores.append(s2)

bars1 = ax.bar(x_bar - width/2, no_pipe_scores, width, color='#94a3b8',
               edgecolor='black', alpha=0.8, label='No Pipeline (numeric only)')
bars2 = ax.bar(x_bar + width/2, with_pipe_scores, width, color='#3b82f6',
               edgecolor='black', alpha=0.8, label='Full Pipeline (numeric + categorical)')

for bar, val in zip(bars1, no_pipe_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontsize=9)
for bar, val in zip(bars2, with_pipe_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontsize=9)

ax.set_xticks(x_bar)
ax.set_xticklabels([n for n, _ in models_compare], fontsize=11)
ax.set_ylabel('CV Accuracy', fontsize=12)
ax.set_title('前處理的影響：Pipeline vs No Pipeline\nImpact of Preprocessing',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('觀察：')
print('- KNN 和 SVM 對特徵縮放敏感 -> Pipeline 帶來顯著提升')
print('- Random Forest 對縮放不敏感 -> 提升主要來自加入類別特徵')

---
## Part 8: Custom Transformer — 自定義轉換器

有時需要自定義的前處理步驟，可繼承 `BaseEstimator` 和 `TransformerMixin`。

**規則：**
- `__init__`: 定義超參數
- `fit`: 從訓練資料學習參數
- `transform`: 套用轉換
- 繼承 `TransformerMixin` 自動獲得 `fit_transform`

In [ ]:
# Part 8: Custom Transformer
class OutlierClipper(BaseEstimator, TransformerMixin):
    """將離群值截斷到指定百分位範圍
    Clips outliers to specified percentile range."""

    def __init__(self, lower_percentile=1, upper_percentile=99):
        self.lower_percentile = lower_percentile
        self.upper_percentile = upper_percentile

    def fit(self, X, y=None):
        self.lower_ = np.nanpercentile(X, self.lower_percentile, axis=0)
        self.upper_ = np.nanpercentile(X, self.upper_percentile, axis=0)
        return self

    def transform(self, X):
        X_clipped = X.copy()
        for j in range(X.shape[1]):
            mask = ~np.isnan(X_clipped[:, j])
            X_clipped[mask, j] = np.clip(X_clipped[mask, j],
                                          self.lower_[j], self.upper_[j])
        return X_clipped


class LogTransformer(BaseEstimator, TransformerMixin):
    """對指定欄位做 log(1+x) 轉換
    Applies log1p transform to reduce skewness."""

    def __init__(self, columns=None):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_transformed = X.copy().astype(float)
        cols = self.columns if self.columns is not None else range(X.shape[1])
        for col in cols:
            mask = ~np.isnan(X_transformed[:, col])
            X_transformed[mask, col] = np.log1p(np.abs(X_transformed[mask, col]))
        return X_transformed


# 示範使用自定義轉換器
print('=== OutlierClipper Demo ===')
data_demo = np.array([[10, 100], [20, 200], [30, 300], [1000, 50000], [25, 250]])
clipper = OutlierClipper(lower_percentile=5, upper_percentile=95)
clipped = clipper.fit_transform(data_demo)
print(f'Before: {data_demo.T}')
print(f'After:  {clipped.T}')
print(f'Lower bounds: {clipper.lower_}')
print(f'Upper bounds: {clipper.upper_}')

# 在 Pipeline 中使用自定義轉換器
print('\n=== Custom Transformer in Pipeline ===')
custom_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clipper', OutlierClipper(lower_percentile=2, upper_percentile=98)),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

scores_custom = cross_val_score(custom_pipe, X_with_missing, target,
                                 cv=5, scoring='accuracy')
print(f'Pipeline with OutlierClipper: {scores_custom.mean():.4f} (+/- {scores_custom.std():.4f})')

# 比較不截斷的 Pipeline
simple_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])
scores_simple = cross_val_score(simple_pipe, X_with_missing, target,
                                 cv=5, scoring='accuracy')
print(f'Pipeline without Clipper:     {scores_simple.mean():.4f} (+/- {scores_simple.std():.4f})')

In [ ]:
# 視覺化 OutlierClipper 的效果
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 使用 income（有離群值）
income_data = income_with_outliers.reshape(-1, 1)
clipper_vis = OutlierClipper(lower_percentile=1, upper_percentile=99)
income_clipped = clipper_vis.fit_transform(income_data)

axes[0].hist(income_data.ravel(), bins=50, color='#dc2626', edgecolor='black',
             alpha=0.5, label='Before Clipping')
axes[0].hist(income_clipped.ravel(), bins=50, color='#2563eb', edgecolor='black',
             alpha=0.5, label='After Clipping')
axes[0].axvline(x=clipper_vis.upper_[0], color='green', linestyle='--',
                label=f'Upper bound={clipper_vis.upper_[0]:.0f}')
axes[0].set_title('OutlierClipper 效果 (Income)\nBefore vs After',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Income')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# LogTransformer 效果
log_trans = LogTransformer(columns=[0])
income_logged = log_trans.fit_transform(income_data)

axes[1].hist(income_data.ravel(), bins=50, color='#dc2626', edgecolor='black',
             alpha=0.5, label='Original', density=True)
axes[1].hist(income_logged.ravel(), bins=50, color='#10b981', edgecolor='black',
             alpha=0.5, label='Log Transformed', density=True)
axes[1].set_title('LogTransformer 效果 (Income)\nOriginal vs Log',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Density')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('自定義 Transformer 可以無縫整合到 Pipeline 中')
print('只要實作 fit() 和 transform() 方法即可')

---
## Exercises — 練習題

### 練習 1：建構混合型資料的完整 Pipeline

使用本週建立的混合型資料集，建構一個完整的 Pipeline：
1. 數值特徵: `SimpleImputer(median)` -> `RobustScaler`
2. 有序類別 (education): `OrdinalEncoder`
3. 名義類別 (city, gender): `OneHotEncoder`
4. 分類器: `GradientBoostingClassifier`

用 5-fold CV 評估效能。

**提示：** 使用 `ColumnTransformer` 分別處理三種欄位。

In [ ]:
# 練習 1：Complete Pipeline for Mixed Data
# TODO: 定義數值 Pipeline (imputer -> RobustScaler)
# TODO: 定義有序類別 Pipeline (OrdinalEncoder)
# TODO: 定義名義類別 Pipeline (OneHotEncoder)
# TODO: 用 ColumnTransformer 組合
# TODO: Pipeline(preprocessor, GradientBoostingClassifier)
# TODO: cross_val_score 5-fold CV


### 練習 2：比較有/無特徵工程的模型效能

使用 California Housing 資料集，比較以下兩個 Pipeline 的 5-fold CV R2 分數：
1. 基本: `StandardScaler` -> `LinearRegression`
2. 進階: `PowerTransformer` -> `PolynomialFeatures(degree=2)` -> `StandardScaler` -> `LinearRegression`

**提示：** 使用 `sklearn.linear_model.LinearRegression`。

In [ ]:
# 練習 2：Feature Engineering Impact
# from sklearn.linear_model import LinearRegression
# TODO: 載入 California Housing
# TODO: 建構基本 Pipeline
# TODO: 建構進階 Pipeline (with PolynomialFeatures)
# TODO: 分別計算 5-fold CV R2 score
# TODO: 比較並印出結果


### 練習 3：在 Pipeline 中使用 PolynomialFeatures

使用 Iris 資料集（取前 2 個特徵），比較以下三個 Pipeline 的 5-fold CV 分數：
1. `StandardScaler` -> `LogisticRegression`
2. `StandardScaler` -> `PolynomialFeatures(degree=2)` -> `LogisticRegression`
3. `StandardScaler` -> `PolynomialFeatures(degree=3)` -> `LogisticRegression`

觀察多項式特徵對線性模型的效能提升效果。

**提示：** 設定 `PolynomialFeatures(include_bias=False)` 避免重複截距項。

In [ ]:
# 練習 3：PolynomialFeatures in Pipeline
# TODO: 載入 Iris 資料 (前 2 個特徵)
# TODO: 建構 3 個 Pipeline (degree=1, 2, 3)
# TODO: 用 cross_val_score 比較
# TODO: 印出特徵數量的變化


---
## Summary — 本週重點回顧

### 關鍵概念 Key Takeaways

1. **類別編碼** 依據類別性質選擇：有序用 `OrdinalEncoder`，名義用 `OneHotEncoder`
2. **特徵縮放** 對距離型模型（KNN、SVM、LR）至關重要，但樹模型不受影響
   - StandardScaler: 常態分布首選
   - RobustScaler: 有離群值時首選
   - MinMaxScaler: 需固定範圍時使用，但對離群值最敏感
3. **缺失值處理** 策略多樣：Mean/Median（簡單）、KNN（考慮相關性）
4. **特徵轉換** (Log, Power, Box-Cox) 可改善偏態分布
5. **Pipeline** 確保 fit 只在訓練集進行，避免資料洩漏 (Data Leakage)
6. **ColumnTransformer** 可對不同類型的特徵套用不同的前處理流程
7. **自定義 Transformer** 繼承 `BaseEstimator` + `TransformerMixin` 即可整合到 Pipeline
8. 前處理的影響因模型而異：SVM/KNN 受益最大，樹模型受影響最小

### 下週預告 Next Week Preview
**第 10 週：超參數調校 (Hyperparameter Tuning)** — 
學習 GridSearchCV、RandomizedSearchCV、學習曲線分析、驗證曲線，
以及如何系統性地找到最佳模型配置。